<h1> <center> Obtaining and visualizing a digital elevation model (DEM) </center></h1>

Digital evelation models (DEMs) are used widely in volcanology. 

This notebook demonstrates how to obtain elevation data from OpenTopograhy, a public open data service for DEMs, directly from a Jupyter notebook on VICTOR. See 
https://opentopography.org/ for more information. 

This notebook makes use of the OpenTopograhy API:
https://portal.opentopography.org/apidocs/#/

On VICTOR, we use two main types of file for topographical data.

#### <center> ASCII (.asc) and TIFF/GeoTIFF (.tiff) </center>

ASC is a human-readable text file with no additional/hidden information. If a need for multiple bands or a specific projection are required, additional files must be proved.<br/>TIFF/GeoTIFF is binary encoded, so we cannot read it like normal text, and has additional parameters/properties set in the file.

In [1]:
#Import the required general libraries

import rasterio as rio
import rioxarray as rxr
import xarray as xr
import sys
import pyproj
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np
import urllib.request
import utm
import openpyxl

#Import the VICTOR library
sys.path.insert(0, "/home/jovyan/shared/Libraries")
import victor

# In this cell, we choose a volcano by name, set the size of the DEM we need, and which format and data source it should come from

In [ ]:
volcano = victor.search_volcano('Hood') # Get volcano information from the GVP database

center_lat = volcano[1][0]  # Get volcano latitude
center_lon = volcano[2][0]  # Get volcano longitude

# Set the extent of the region of interest, in meters in each direction from the summit
eastwest_extent = 10000
northsouth_extent = 10000

# Set bounding box in lat/lon
utms = utm.from_latlon(center_lat, center_lon)
[north_lat, east_lon] = utm.to_latlon(utms[0]+eastwest_extent, utms[1]+northsouth_extent, utms[2], utms[3])
[south_lat, west_lon] = utm.to_latlon(utms[0]-eastwest_extent, utms[1]-northsouth_extent, utms[2], utms[3])

# Select a DEM format: Input "ascii" for ASCII format (.asc) and "tiff" for geotiff format
outputFormat = "tiff"

# Enter the name of the output file (without extension)
name = "Hood"

dataset = "SRTMGL1"

print(center_lat, ", ", center_lon) 

## Now, we can use the VICTOR library to download a DEM. We demonstrate using input in either Lat/Lon or UTM coordinate. The output from OpenTopography is always in global coordinates (Lat/Lon).  

In [ ]:
lon = center_lon
lat = center_lat

victor.download_dem(lat+.05,lat-.05,lon+.05,lon-.05,"tiff","COP30")

### victor.py supports giving the target coordinates in UTM. So, let's first determine the UTM coordinates of the volcano, and then get the DEM.

In [ ]:
(utm_east, utm_north, zone, _) = utm.from_latlon(center_lat, center_lon)
print(utm_east, ",", utm_north, ",", zone)

In [ ]:
victor.download_dem_utm(utm_north+6000,utm_north-5000,utm_east+5000,utm_east-5000, hemisphere='S',utm_zone=zone, dataset="COP30", outputFormat="tiff")

## Let's download a DEM for a segment of the eastern Snake River Plane

In [ ]:
lat_ESRP = 43.489687
lon_ESRP = -112.733177

(esrp_utm_east, esrp_utm_north, zone, _) = utm.from_latlon(lat_ESRP, lon_ESRP)
print(esrp_utm_east, ",", esrp_utm_north, ",", zone)

#1) Get a Lat/Long DEM in ASC and TIFF
esrp_tiff_name = victor.download_dem(lat_ESRP+.15,lat_ESRP-.15,lon_ESRP+.15,lon_ESRP-.15, "tiff", 'COP30')
esrp_asc_name = victor.download_dem(lat_ESRP+.15,lat_ESRP-.15,lon_ESRP+.15,lon_ESRP-.15, "ascii", 'COP30')


## OpenTopography holds higher resolution (10m) topograhy data for some areas in the US. The next cell demonstrate using it for an area in the eastern Snake River Plane

In [ ]:
#victor.download_dem_usgs(40.288, 40.234, -105.583, -105.673, "ascii", '10m')

victor.download_dem_usgs(lat_ESRP+.05,lat_ESRP-.05,lon_ESRP+.05,lon_ESRP-.05, "ascii", '10m')

## The files download in latitude/longitude, but many of the modeling tools require the topography in UTM. Lets convert here:

In [3]:
victor.convert_dem_format(input_tiff="/home/jovyan/shared/DEMs/NorthAmerica/ESRP.geotiff", output_path="ESRP_utm.tif", output_format="geotiff", utm_zone=12) #Put your desired output file name here


Opening /home/jovyan/shared/DEMs/NorthAmerica/ESRP.geotiff...
Input CRS is geographic: EPSG:4326
Reprojecting to UTM zone 12 north (CRS: EPSG:32612)
Reprojecting to UTM...
Saving UTM GeoTIFF: ESRP_utm.tif
Conversion complete. Reprojected GEOTIFF file saved to: ESRP_utm.tif


'ESRP_utm.tif'

## Next, we can plot the DEM we just downloaded
We use a plotting function from the victor.py library.

In [ ]:
victor.plot_dem("ESRP_utm.asc", markercoords = np.array([esrp_utm_east,esrp_utm_north]))